# Notebook 01 — Carregamento e Análise Exploratória Inicial

**Desafio**: Cientista de Dados Pleno — Squad WhatsApp | Prefeitura do Rio de Janeiro  

## Objetivo

Este notebook realiza o carregamento dos dados brutos do bucket GCS e conduz uma análise exploratória inicial para entender:

1. Estrutura e qualidade das duas tabelas principais
2. Distribuição dos status de disparo (DELIVERED, READ, FAILED, SENT)
3. Perfil dos telefones cadastrados (tipo, qualidade, quantidade de sistemas)
4. Estrutura do campo array `telefone_aparicoes` — o coração do problema de multiplicidade

---

## 0. Setup e Importações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pyarrow.parquet as pq
import gcsfs

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

BUCKET = 'gs://case_vagas/whatsapp'

print('Ambiente configurado com sucesso.')

## 1. Carregamento dos Dados

Os dados estão disponíveis em um bucket público do Google Cloud Storage. Utilizamos `gcsfs` para leitura direta sem necessidade de download local.

In [ ]:
fs = gcsfs.GCSFileSystem(token='anon')

# Listar arquivos disponíveis
arquivos = fs.ls(BUCKET.replace('gs://', ''))
print('Arquivos no bucket:')
for f in arquivos:
    size = fs.info(f)['size'] / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# Carregar tabela de disparos
with fs.open(f"{BUCKET.replace('gs://', '')}/base_disparo_mascarado") as f:
    df_disparo = pd.read_parquet(f)

print(f'base_disparo_mascarado: {df_disparo.shape[0]:,} linhas × {df_disparo.shape[1]} colunas')

In [ ]:
# Carregar tabela de dimensão de telefones
with fs.open(f"{BUCKET.replace('gs://', '')}/dim_telefone_mascarado") as f:
    df_tel = pd.read_parquet(f)

print(f'dim_telefone_mascarado: {df_tel.shape[0]:,} linhas × {df_tel.shape[1]} colunas')

## 2. Inspeção do Schema

### 2.1 Tabela de Disparos (`base_disparo_mascarado`)

In [ ]:
df_disparo.dtypes.to_frame('dtype').assign(
    nulos=df_disparo.isna().sum(),
    pct_nulo=(df_disparo.isna().mean() * 100).round(2),
    nunique=df_disparo.nunique()
)

In [ ]:
df_disparo.head(3)

### 2.2 Tabela de Dimensão de Telefones (`dim_telefone_mascarado`)

In [ ]:
# Colunas sem o campo array (para análise inicial)
cols_simples = [c for c in df_tel.columns if c != 'telefone_aparicoes']

df_tel[cols_simples].dtypes.to_frame('dtype').assign(
    nulos=df_tel[cols_simples].isna().sum(),
    pct_nulo=(df_tel[cols_simples].isna().mean() * 100).round(2),
    nunique=df_tel[cols_simples].nunique()
)

In [ ]:
df_tel[cols_simples].head(3)

### 2.3 Explorando o campo array `telefone_aparicoes`

Este campo é o mais rico — contém a história de cada telefone em diferentes sistemas da Prefeitura. Cada registro é uma lista de structs com `id_sistema`, `cpf`, `proprietario_tipo` e `registro_data_atualizacao`.

In [ ]:
# Verificar o tipo e um exemplo do campo array
print('Tipo da coluna:', type(df_tel['telefone_aparicoes'].iloc[0]))
print('\nExemplo de um registro:')
print(df_tel['telefone_aparicoes'].iloc[0])

In [ ]:
# Explodir o array para análise
df_aparicoes = df_tel[['telefone_mascarado', 'telefone_aparicoes']].explode('telefone_aparicoes').reset_index(drop=True)

# Normalizar os structs para colunas
df_aparicoes_norm = pd.json_normalize(df_aparicoes['telefone_aparicoes'])
df_aparicoes = pd.concat([df_aparicoes[['telefone_mascarado']], df_aparicoes_norm], axis=1)

print(f'Após explode: {df_aparicoes.shape[0]:,} linhas')
print(f'Colunas: {list(df_aparicoes.columns)}')
df_aparicoes.head(5)

## 3. Análise da Tabela de Disparos

### 3.1 Distribuição de Status de Disparo

In [ ]:
status_counts = df_disparo['status_disparo'].value_counts()
status_pct = df_disparo['status_disparo'].value_counts(normalize=True) * 100

status_df = pd.DataFrame({'count': status_counts, 'pct': status_pct.round(2)})
print('Distribuição de status_disparo:\n')
print(status_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume absoluto
cores = {'DELIVERED': '#2ecc71', 'READ': '#3498db', 'FAILED': '#e74c3c', 'SENT': '#f39c12'}
status_counts.plot(kind='bar', ax=axes[0],
                   color=[cores.get(s, '#95a5a6') for s in status_counts.index],
                   edgecolor='white')
axes[0].set_title('Volume de Disparos por Status', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_xticklabels(status_counts.index, rotation=0)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

for i, (v, p) in enumerate(zip(status_counts, status_pct)):
    axes[0].text(i, v + status_counts.max() * 0.01, f'{p:.1f}%', ha='center', fontsize=10)

# Pizza
axes[1].pie(status_counts, labels=status_counts.index,
            colors=[cores.get(s, '#95a5a6') for s in status_counts.index],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporção por Status', fontweight='bold')

plt.suptitle('Status dos Disparos WhatsApp', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Taxa de sucesso consolidada
taxa_sucesso = status_pct.get('DELIVERED', 0) + status_pct.get('READ', 0)
print(f'\nTaxa de sucesso (DELIVERED + READ): {taxa_sucesso:.2f}%')

### 3.2 Motivos de Falha

In [ ]:
falhas = df_disparo[df_disparo['status_disparo'] == 'FAILED']
falhas_desc = falhas['descricao_falha'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 5))
falhas_desc.plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
ax.set_title('Top 10 Motivos de Falha nos Disparos', fontweight='bold')
ax.set_xlabel('Quantidade')
ax.invert_yaxis()

for i, v in enumerate(falhas_desc):
    ax.text(v + falhas_desc.max() * 0.01, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nTotal de falhas: {len(falhas):,} ({len(falhas)/len(df_disparo)*100:.1f}% dos disparos)')

### 3.3 Volume de Disparos ao Longo do Tempo

In [ ]:
# Converter timestamps
df_disparo['criacao_envio_datahora'] = pd.to_datetime(df_disparo['criacao_envio_datahora'])

# Volume diário
volume_diario = df_disparo.groupby(df_disparo['criacao_envio_datahora'].dt.date).size()

fig, ax = plt.subplots(figsize=(14, 4))
volume_diario.plot(ax=ax, color='#3498db', linewidth=1.5)
ax.fill_between(volume_diario.index, volume_diario.values, alpha=0.2, color='#3498db')
ax.set_title('Volume Diário de Disparos', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Nº de Disparos')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
plt.tight_layout()
plt.show()

print(f'Período: {volume_diario.index.min()} a {volume_diario.index.max()}')
print(f'Média diária: {volume_diario.mean():.0f} disparos/dia')

## 4. Análise da Dimensão de Telefones

### 4.1 Tipo e Qualidade dos Telefones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Tipo de telefone
tipo_vc = df_tel['telefone_tipo'].value_counts()
axes[0].pie(tipo_vc, labels=tipo_vc.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(tipo_vc)), startangle=90)
axes[0].set_title('Tipo de Telefone', fontweight='bold')

# Qualidade do telefone
qual_vc = df_tel['telefone_qualidade'].value_counts()
cores_qual = {'ALTA': '#2ecc71', 'MEDIA': '#f39c12', 'BAIXA': '#e74c3c'}
qual_vc.plot(kind='bar', ax=axes[1],
             color=[cores_qual.get(q, '#95a5a6') for q in qual_vc.index],
             edgecolor='white')
axes[1].set_title('Qualidade do Telefone (score interno)', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_xticklabels(qual_vc.index, rotation=0)

for i, v in enumerate(qual_vc):
    axes[1].text(i, v + qual_vc.max() * 0.01,
                 f'{v/len(df_tel)*100:.1f}%', ha='center', fontsize=10)

# Validação do telefone
val_vc = df_tel['validacao_telefone'].value_counts()
val_vc.plot(kind='bar', ax=axes[2],
            color=sns.color_palette('Set2', len(val_vc)),
            edgecolor='white')
axes[2].set_title('Status de Validação', fontweight='bold')
axes[2].set_xlabel('')
axes[2].set_xticklabels(val_vc.index, rotation=30, ha='right')

plt.suptitle('Perfil da Base de Telefones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Multiplicidade: Quantos Sistemas por Telefone?

Este é o coração do problema: um telefone pode aparecer em múltiplos sistemas, criando redundância mas também complexidade na escolha.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de sistemas por telefone
sys_counts = df_tel['telefone_sistemas_quantidade'].value_counts().sort_index()
sys_counts.plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Distribuição: Nº de Sistemas por Telefone', fontweight='bold')
axes[0].set_xlabel('Quantidade de Sistemas')
axes[0].set_ylabel('Nº de Telefones')
axes[0].set_xticklabels(sys_counts.index, rotation=0)

for i, v in enumerate(sys_counts):
    axes[0].text(i, v + sys_counts.max() * 0.01,
                 f'{v/len(df_tel)*100:.1f}%', ha='center', fontsize=9)

# Distribuição de proprietários por telefone
prop_counts = df_tel['telefone_proprietarios_quantidade'].value_counts().sort_index()
prop_counts.plot(kind='bar', ax=axes[1], color='#e67e22', edgecolor='white')
axes[1].set_title('Distribuição: Nº de Proprietários por Telefone', fontweight='bold')
axes[1].set_xlabel('Quantidade de Proprietários')
axes[1].set_ylabel('Nº de Telefones')
axes[1].set_xticklabels(prop_counts.index, rotation=0)

plt.suptitle('Multiplicidade na Base de Telefones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Telefones em apenas 1 sistema: {(df_tel["telefone_sistemas_quantidade"] == 1).sum():,} ({(df_tel["telefone_sistemas_quantidade"] == 1).mean()*100:.1f}%)')
print(f'Telefones em 2+ sistemas: {(df_tel["telefone_sistemas_quantidade"] >= 2).sum():,} ({(df_tel["telefone_sistemas_quantidade"] >= 2).mean()*100:.1f}%)')
print(f'Telefones com 2+ proprietários: {(df_tel["telefone_proprietarios_quantidade"] >= 2).sum():,} ({(df_tel["telefone_proprietarios_quantidade"] >= 2).mean()*100:.1f}%)')

### 4.3 Sistemas de Origem Presentes na Base

In [ ]:
sistema_counts = df_aparicoes['id_sistema'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
sistema_counts.plot(kind='bar', ax=ax, color='#9b59b6', edgecolor='white')
ax.set_title('Volume de Aparições por Sistema de Origem', fontweight='bold')
ax.set_xlabel('Sistema')
ax.set_ylabel('Nº de Aparições')
ax.set_xticklabels(sistema_counts.index, rotation=45, ha='right')

for i, v in enumerate(sistema_counts):
    ax.text(i, v + sistema_counts.max() * 0.01,
            f'{v/sistema_counts.sum()*100:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nTotal de sistemas distintos: {df_aparicoes["id_sistema"].nunique()}')
print(f'Total de aparições (após explode): {len(df_aparicoes):,}')

## 5. Cobertura do Join

Quantos disparos conseguimos associar a um telefone na dimensão?

In [ ]:
telefones_disparo = set(df_disparo['contato_telefone'].dropna().unique())
telefones_dim = set(df_tel['telefone_mascarado'].unique())

intersecao = telefones_disparo & telefones_dim
so_disparo = telefones_disparo - telefones_dim
so_dim = telefones_dim - telefones_disparo

print('=== Cobertura do Join ===')
print(f'Telefones na base de disparos:     {len(telefones_disparo):>10,}')
print(f'Telefones na dimensão:             {len(telefones_dim):>10,}')
print(f'Interseção (join perfeito):        {len(intersecao):>10,}  ({len(intersecao)/len(telefones_disparo)*100:.1f}% dos disparos)')
print(f'Só na base de disparos (sem dim):  {len(so_disparo):>10,}  ({len(so_disparo)/len(telefones_disparo)*100:.1f}%)')
print(f'Só na dimensão (nunca disparado):  {len(so_dim):>10,}')

# Impacto em número de linhas
disparos_com_dim = df_disparo['contato_telefone'].isin(telefones_dim).sum()
print(f'\nDisparos com match na dimensão: {disparos_com_dim:,} ({disparos_com_dim/len(df_disparo)*100:.1f}%)')

## 6. Conclusões da EDA Inicial

**Achados principais:**

1. **Status de disparos**: A distribuição entre DELIVERED/READ/FAILED/SENT indica a linha de base que nosso algoritmo deve superar.

2. **Motivos de falha**: O principal motivo de falha revela se o problema é "número inativo no WhatsApp" (número ruim) ou problemas técnicos de envio. Isso confirma a hipótese de que a qualidade do dado de telefone impacta diretamente a entrega.

3. **Multiplicidade**: Uma parcela dos telefones aparece em múltiplos sistemas — é exatamente para esses casos que o algoritmo de priorização se torna crítico.

4. **Viés de seleção**: Alguns sistemas dominam em volume de aparições. No próximo notebook, investigaremos se esse volume alto reflete real qualidade ou apenas preferência histórica do motor de disparos.

5. **Cobertura do join**: A maioria dos disparos tem correspondência na dimensão de telefones, permitindo análise robusta.

---

**Próximo passo**: Notebook 02 — Análise de Qualidade por Sistema de Origem e Decaimento Temporal.